## 문제 1: [통계/회귀] 센서 온도 기반 마모도 예측 (권장 30분)

```sensor_regression.csv```를 불러오세요. 온도(```Temperature```) 센서 값에 간헐적으로 결측치(NaN)가 있습니다.

1. 결측치가 있는 행(row)을 완전히 삭제하세요. (힌트: ```dropna()```)

2. 온도를 `X`로, 마모도(`Wear_Level`)를 y로 설정하여 `LinearRegression` 모델을 학습시키세요.

3. 학습된 모델의 **$R^2$ Score(결정계수)**를 계산하여 출력하세요.

In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv('C:\Programmers\Gemini_example\data\sensor_regression.csv')
def solve_prob1(df):
    df = df.dropna()
    # X,y 분리 및 X 차원 변경(2차원)
    X = df['Temperature'].values.reshape(-1,1)
    y = df['Wear_Level'].values
    
    # 모델 학습 및 평가
    model = LinearRegression()
    model.fit(X,y)
    y_pred = model.predict(X)
    
    print(f"[문제 1] R2 Score: {r2_score(y, y_pred):.4f}")
    
solve_prob1(df)

[문제 1] R2 Score: 0.9822


## 문제 2: [비전/데이터로더] 불량 이미지 메타데이터 파싱 (권장 40분)

```defect_labels.csv```를 불러오세요. 실제 이미지는 없지만, 이 CSV 파일을 읽어 PyTorch 모델에 데이터를 공급하는 ```Dataset``` 클래스를 작성해야 합니다. 

1. '정상'은 0, '스크래치'는 1, '찍힘'은 2로 라벨을 숫자로 변환하세요.
2. PyTorch의 Dataset을 상속받아 ```DefectDataset``` 클래스를 만드세요.
3. ```__getitem__``` 메서드에서는 실제 이미지를 불러오는 대신, torch.randn(3, 224, 224)를 사용하여 가짜(Mock) 이미지 텐서를 생성하고 변환된 숫자 라벨과 함께 반환하세요.

4. ```DataLoader```에 태워서 첫 번째 배치의 데이터 Shape을 출력해 보세요.

In [3]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



class DefectDataset(Dataset):
    def __init__(self, csv_file):
        self.df = pd.read_csv("C:\Programmers\Gemini_example\data\defect_labels.csv")
        # 라벨 인코딩
        label_map = {'정상':0, '스크래치':1, '찍힘':2}
        self.df['defect_type'] = self.df['defect_type'].map(label_map)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # 실제 환경에서는: img = Image.open(self.df.iloc[idx]['image_file'])
        # 코테 환경에서는 가짜 텐서로 대체하여 파이프라인만 검증
        mock_image = torch.randn(3, 224, 224)
        label = self.df.iloc[idx]['defect_type']
        
        return mock_image, torch.tensor(label, dtype = torch.long)

def solve_prob2():
    dataset = DefectDataset("C:\Programmers\Gemini_example\data\defect_labels.csv")
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
    
    images, labels = next(iter(dataloader))
    print(f"[문제 2] 이미지 배치 형태: {images.shape}")
    print(f"[문제 2] 라벨 배치 형태: {labels.shape}")   


solve_prob2()    

[문제 2] 이미지 배치 형태: torch.Size([2, 3, 224, 224])
[문제 2] 라벨 배치 형태: torch.Size([2])


## 문제 3: [시계열/파이프라인] TACO 시스템 다변량 예측 (권장 50분)

```taco_timeseries.csv```를 불러오세요. 과거 3분(Window=3)의 데이터를 보고, 다음 1분의 진동(Vibration) 수치를 예측하기 위한 전처리를 수행하세요.


1. ```Temp``` 컬럼의 결측치를 바로 직전 시간의 값으로 채우세요.

2. ```Machine_Status``` 컬럼의 문자열을 숫자(ON:0, WARN:1, ERROR:2)로 변환하세요.

3. ```Time``` 컬럼은 제외하고 ```Temp, Vibration, Machine_Status``` 3개 컬럼을 MinMaxScaler로 0~1 사이로 스케일링하세요.

4. Window Size = 3으로 슬라이딩 윈도우를 적용하여 PyTorch 텐서 ```X```(형태: Batch, 3, 3)와 ```y```(형태: Batch, 1)를 생성하세요.


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import torch

def solve_prob3(df):
    df = pd.read_csv('C:\Programmers\Gemini_example\data/taco_timeseries.csv')
    df['Temp'] = df['Temp'].ffill()
    df['Machine_Status'] = df['Machine_Status'].map({'ON':0, 'WARN':1, 'ERROR':2})
    
    # Time 컬럼 버리고 피처만 추출
    features = df[['Temp', 'Vibration', 'Machine_Status']].values
    
    # 스케일링
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(features)
    
    # sliding window
    window_size = 3
    target_idx = 1 # 진동은 1번 인덱스라서
    
    X, y = [], []
    for i in range(len(scaled_data) - window_size):
        X.append(scaled_data[i : i + window_size])
        y.append(scaled_data[i + window_size, target_idx])
    
    
    X_tensor = torch.tensor(np.array(X), dtype=torch.float32)
    y_tensor = torch.tensor(np.array(y), dtype=torch.float32).view(-1,1)
        
    print(f"[문제 3] X 형태: {X_tensor.shape}")
    print(f"[문제 3] y 형태: {y_tensor.shape}")   
    
solve_prob3(df)
    

[문제 3] X 형태: torch.Size([7, 3, 3])
[문제 3] y 형태: torch.Size([7, 1])


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler


def prepare_data(window_size = 3):
    

In [ ]:
# 모델 정의

class SensorLSTM(nn.Module):
    def __init__(self, input_features, hidden_size = 32):
        super(SensorLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size = input_features, hidden_size=hidden_size,batch_size=True)
        self.fc = nn.Linear(hidden_size,1)
    
    def forward(self, X):
        lstm_out, _ = self.lstm(X)
        last_time_step = lstm_out[:, -1, :]
        return self.fc(last_time_step)
    
    